## Explore the subsampling of the AGBD dataset

`env: dwn`

Requirements:
* 10-20 GB
* provide AGB and CH labels
* drop the features that are really irrelevant
* maybe drop the existing structure, and just do the train/val/test splits


### Imports and global variables definition

In [ ]:
from os.path import join, isfile
import geopandas as gpd
import numpy as np
import pickle

In [ ]:
path_to_h5 = '/scratch3/gsialelli/patches'
h5_fnames = [join(path_to_h5, f'data_subset-{year}-v4_{i}-20.h5') for i in range(20) for year in [2019, 2020]] 
path_shp = "/scratch3/gsialelli/BiomassDatasetCreation/Data/download_Sentinel/sentinel_2_index_shapefile.shp"
path_geojson = join('/scratch3', 'gsialelli', 'BiomassDatasetCreation', 'Data', 'countrySelection', 'AOIs.geojson')
all_s2_tiles = gpd.read_file(path_shp, engine = 'pyogrio').drop_duplicates(subset = ['Name'])

In [ ]:
rng = np.random.default_rng(seed=1)

### Get the current distributions per region

In [ ]:
# List of regions in the dataset
regions = ['California', 'Cuba', 'Paraguay', 'UnitedRepublicofTanzania', 'Ghana', 'Austria', 'Greece', 'Nepal', 'ShaanxiProvince', 'NewZealand', 'FrenchGuiana']

# Get the tiles for each region
countries_geojson = gpd.read_file(path_geojson)
tiles_per_region = {}
for region in regions:
    country_geojson = countries_geojson[countries_geojson['name'].isin([region])]
    country_tiles = all_s2_tiles[all_s2_tiles.geometry.intersects(country_geojson.geometry.values[0])]
    tiles_per_region[region] = country_tiles['Name'].tolist()

# Save the mapping
with open('tiles_per_region.pkl', 'wb') as f: 
    pickle.dump(tiles_per_region, f)

In [ ]:
def get_regional_distribution(region, tiles_per_region, h5_fnames, bins, biomes) :
    """
    This function returns the distribution of biomass bins and biomes for the specified region.

    Args:
    - region (str): The name of the region.
    - tiles_per_region (dict): A dictionary mapping regions to their corresponding list of tiles.
    - h5_fnames (list): A list of HDF5 file paths containing the dataset.
    - bins (list): A list of biomass bins.
    - biomes (list): A list of biome categories.

    Returns:
    - agb_cum_dist (dict): A dictionary with biomass bin ranges as keys and their cumulative counts as values.
    - biome_cum_dist (dict): A dictionary with biome categories as keys and their cumulative counts as values.
    """

    # Initialize placeholders
    lbs, ubs = bins[:-1], bins[1:]
    agb_cum_dist = {f'{lb}-{ub}': 0 for lb, ub in zip(lbs, ubs)}
    biome_cum_dist = {biome: 0 for biome in biomes}

    # Iterate over the footprints in the region
    region_tiles = tiles_per_region[region]
    for fname in h5_fnames :
        with h5py.File(fname, 'r') as f:
            f_tiles = np.intersect1d(list(f.keys()), region_tiles)
            if len(f_tiles) == 0 : continue
            for tile in f_tiles :
                # Load AGB and biome data
                agb_data = f[tile]['GEDI']['agbd'][:]
                biome_data = f[tile]['LC'][:, 12, 12, 0]
                # Update cumulative distributions
                for lb, ub in zip(lbs, ubs): agb_cum_dist[f'{lb}-{ub}'] += np.sum((agb_data >= lb) & (agb_data < ub))
                for biome in biomes: biome_cum_dist[biome] += np.sum(biome_data == biome)
    
    return agb_cum_dist, biome_cum_dist



In [ ]:
# For each region, get the AGBD binned distribution and biome distribution

# Biomass bins
bins = np.arange(0, 501, 10)

# Biomes
biomes = [20, 30, 40, 90, 111, 112, 114, 115, 116, 121, 122, 124, 125, 126]

# Iterate over the regions
results = {}
for region in regions:
    print(region)
    agb_cum_dist, biome_cum_dist = get_regional_distribution(region, tiles_per_region, h5_fnames, bins, biomes)
    results[region] = {'agb': agb_cum_dist, 'biome': biome_cum_dist}


In [ ]:
# Save the results
if not isfile('regional_distributions.pkl'):
    with open('regional_distributions.pkl', 'wb') as f:
        pickle.dump(results, f)